# ASSIGNMENT 11
## Fine-Tuning GPT-2 on a Custom Text Style

**Generative AI**  
**Name:** Lakhan Singh  
**Model:** GPT-2 (`gpt2`)

### Objective

In this assignment, I fine-tune a pretrained GPT-2 model on a small custom text dataset. The main purpose is to see whether GPT-2 can adapt its vocabulary, tone, and sentence patterns to a particular writing style.

I use the same prompt that I used in Assignment 2 so that I can compare the original GPT-2 output with the output produced after fine-tuning.

## 1. Assignment Requirements

For this experiment, I follow these main steps:

- Prepare approximately 200–300 lines of text with a consistent writing style.
- Clean the text so that it contains only useful language content.
- Load the pretrained GPT-2 model.
- Tokenize the custom dataset.
- Fine-tune GPT-2 using the Hugging Face `Trainer`.
- Generate text from the original GPT-2 model.
- Generate text from the fine-tuned GPT-2 model.
- Use the same prompt for both models.
- Compare the outputs and identify at least two clear differences.

I chose an original technology and AI-focused writing style for the dataset. The text is intentionally written in a consistent, simple explanatory and motivational style.

## 2. Install Required Libraries

I use Hugging Face Transformers for GPT-2 and the `Trainer` API, PyTorch for model training, and Pandas for organizing the final comparison.

The versions below are suitable for running this notebook in Google Colab.

In [24]:
!pip install -q -U transformers datasets accelerate pandas matplotlib

## 3. Import Libraries

Here I import the libraries needed for the complete experiment. I also set a random seed so that the experiment is easier to reproduce.

In [25]:
import os
import random
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForLanguageModeling,
    Trainer,
    TrainingArguments,
    set_seed
)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

print("Libraries imported successfully.")
print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

Libraries imported successfully.
PyTorch version: 2.11.0+cpu
CUDA available: False


## 4. Prepare the Custom Dataset

For this assignment, I created an original dataset of 250 short lines about artificial intelligence, technology, learning, creativity, and the future of work.

I kept the writing style fairly consistent. Most lines use short explanatory sentences with a positive and practical tone. This gives GPT-2 a recognizable style to learn.

Because the assignment asks for around 200–300 lines, 250 lines gives enough examples for the experiment while still keeping the dataset small.

In [26]:
dataset_text = [
    "Artificial intelligence is changing the way people solve everyday problems.",
    "Technology becomes more useful when people understand how to use it responsibly.",
    "Machine learning helps computers learn patterns from examples instead of fixed rules.",
    "A good model can turn a large amount of information into useful predictions.",
    "The real value of artificial intelligence comes from solving meaningful problems.",
    "Learning AI requires curiosity, patience, and regular practice.",
    "Small experiments can teach important lessons about complex machine learning systems.",
    "A simple idea can become powerful when it is tested with real data.",
    "Data gives an intelligent system the examples it needs to recognize patterns.",
    "Good data preparation is often as important as choosing a sophisticated model.",
    "A model learns from the information that we provide during training.",
    "Better training data can improve the quality and reliability of predictions.",
    "Artificial intelligence can support people without replacing human judgment.",
    "Human creativity becomes stronger when technology is used as a helpful tool.",
    "The future of work will require people to understand both technology and communication.",
    "People who keep learning can adapt to changes in technology more easily.",
    "Programming is not only about writing code but also about thinking clearly.",
    "Debugging teaches patience because useful solutions often appear after several attempts.",
    "Every error in a program can become a lesson when it is understood carefully.",
    "A strong technical foundation makes advanced concepts easier to learn.",
    "Python is useful because its simple syntax allows beginners to focus on logic.",
    "Machine learning combines mathematics, programming, data, and experimentation.",
    "Deep learning uses neural networks to learn complicated representations from data.",
    "Neural networks improve by adjusting their parameters during training.",
    "Training a model is a process of repeated learning and correction.",
    "The first result is rarely the final result in a machine learning project.",
    "Experimentation helps us understand which decisions improve a model.",
    "A clear objective makes an AI project easier to design and evaluate.",
    "Good documentation makes technical work easier to understand and reproduce.",
    "A useful project should explain not only what was built but also why it was built.",
    "Artificial intelligence can help people process information faster.",
    "Automation can reduce repetitive work and give people more time for creative tasks.",
    "The best automation removes unnecessary effort without removing human responsibility.",
    "Technology should be designed around real needs rather than trends alone.",
    "Responsible AI requires attention to privacy, fairness, safety, and reliability.",
    "A powerful model is not automatically a trustworthy model.",
    "AI systems should be tested before they are used in important situations.",
    "Evaluation helps us understand where a model performs well and where it fails.",
    "Failure cases are valuable because they reveal weaknesses in a system.",
    "A model that works on one dataset may behave differently on another dataset.",
    "Generalization is important because real-world data is rarely identical to training data.",
    "Overfitting happens when a model learns the training examples too closely.",
    "A balanced dataset can help a model learn more useful patterns.",
    "Simple baselines are useful because they give us something to compare against.",
    "A complex model should provide a meaningful improvement over a simpler approach.",
    "Artificial intelligence is most effective when technical decisions match the problem.",
    "Good problem definition comes before model selection.",
    "Before training a model, I first ask what result I actually need.",
    "Clear questions usually lead to clearer experiments.",
    "Learning technology is easier when difficult topics are divided into smaller parts.",
    "Consistent practice is more useful than studying a difficult topic only once.",
    "A learner improves by building, testing, reviewing, and rebuilding.",
    "Projects turn theoretical knowledge into practical understanding.",
    "A working project is also a record of what a learner has understood.",
    "Building from scratch reveals problems that tutorials sometimes hide.",
    "Reading documentation teaches how tools behave beyond simple examples.",
    "Good developers learn to search for answers instead of memorizing every detail.",
    "The ability to investigate a problem is an important technical skill.",
    "A strong engineer does not fear errors but investigates them systematically.",
    "Logs can reveal what a program is doing when the final output looks confusing.",
    "Testing helps catch mistakes before they become larger problems.",
    "Small tests make large systems easier to maintain.",
    "Reliable software comes from careful design and repeated testing.",
    "Artificial intelligence projects also need engineering discipline.",
    "A model is only one part of a complete AI application.",
    "A useful AI system also needs data pipelines, interfaces, storage, and monitoring.",
    "Deployment is different from experimentation because real users introduce new conditions.",
    "Real-world systems must handle unexpected inputs gracefully.",
    "Performance should be measured using meaningful metrics rather than assumptions.",
    "Accuracy is useful, but it is not the only measure of a good system.",
    "Latency matters when people expect a system to respond quickly.",
    "Memory and computing resources also affect how a model can be deployed.",
    "Efficient models can make intelligent applications more practical.",
    "Cloud computing makes many machine learning resources easier to access.",
    "Local computing can be useful when privacy or offline access is important.",
    "The right infrastructure depends on the requirements of the application.",
    "Artificial intelligence is becoming part of many different industries.",
    "Healthcare can use AI to support analysis and decision making.",
    "Education can use AI to personalize learning experiences.",
    "Transportation can use intelligent systems to improve planning and safety.",
    "Agriculture can use data to improve resource management and crop monitoring.",
    "Finance can use machine learning to identify patterns in large datasets.",
    "Creative tools can use AI to help people explore new ideas.",
    "AI can assist researchers by finding patterns that are difficult to see manually.",
    "The usefulness of AI depends on how well it fits the needs of its users.",
    "Technology should make difficult work clearer, faster, or more accessible.",
    "A good interface can make a powerful model easier for ordinary users to understand.",
    "Human-centered design is important for intelligent applications.",
    "People should understand the limitations of an AI system before trusting its output.",
    "Transparency can improve confidence in automated decisions.",
    "Explainability is especially important when AI affects important decisions.",
    "Privacy should be considered from the beginning of an AI project.",
    "Sensitive information should be handled carefully and securely.",
    "Security is part of responsible software development.",
    "A model can be technically impressive and still create practical risks.",
    "Responsible development requires both technical and social awareness.",
    "The future of AI will depend on people who can combine these perspectives.",
    "Generative AI can create text, images, audio, and other forms of content.",
    "Language models learn relationships between words and tokens from large collections of text.",
    "GPT-2 generates text by predicting the next token from the previous context.",
    "Text generation is influenced by both the model and the decoding settings.",
    "Temperature changes how random the token selection becomes.",
    "Top-k limits the next-token choices to a fixed number of likely tokens.",
    "Top-p selects tokens from a probability mass rather than a fixed count.",
    "Repetition penalties can reduce repeated patterns in generated text.",
    "Generation settings can change the personality of an output without changing the model weights.",
    "Fine-tuning is different because it changes the model parameters using additional training data.",
    "Fine-tuning allows a pretrained model to adapt to a narrower domain or style.",
    "A pretrained model already contains general language knowledge before fine-tuning.",
    "The purpose of fine-tuning is to specialize some of that existing knowledge.",
    "A small dataset can be enough to demonstrate a noticeable stylistic change.",
    "A small dataset can also cause overfitting if training is too aggressive.",
    "This is why training settings should be chosen carefully.",
    "The number of epochs affects how many times the model sees the training data.",
    "Too many epochs may make the model memorize the training examples.",
    "Too few epochs may produce only a small stylistic change.",
    "Learning rate controls how strongly model parameters are updated during training.",
    "A very high learning rate can make training unstable.",
    "A lower learning rate usually gives the model smaller and more controlled updates.",
    "Batch size affects how many examples are processed together during training.",
    "Training time depends on the model, dataset, hardware, and training settings.",
    "GPU acceleration can make transformer training much faster.",
    "Even a small experiment can demonstrate the main idea of transfer learning.",
    "Transfer learning starts with knowledge learned from a larger dataset.",
    "Fine-tuning then adapts that knowledge to a more specific task.",
    "This approach is often more practical than training a large language model from scratch.",
    "The same principle appears in many computer vision and natural language applications.",
    "A model can learn general representations first and specialized behavior later.",
    "Good fine-tuning begins with a clear target behavior.",
    "The training examples should represent the style that we want the model to learn.",
    "Inconsistent examples make it harder for a model to learn one recognizable style.",
    "Clean text reduces unnecessary noise during language model training.",
    "Timestamps and usernames do not usually contribute to a writing style.",
    "Formatting noise can also become an unwanted pattern for the model.",
    "Dataset quality matters even when the dataset is small.",
    "A carefully prepared small dataset can be more useful than a noisy larger dataset.",
    "The goal of this experiment is style adaptation rather than general language learning.",
    "I want the fine-tuned model to sound more like the examples in my dataset.",
    "The base model should provide a useful reference for comparison.",
    "Using the same prompt makes the comparison more meaningful.",
    "If both models receive the same prompt, the difference is easier to observe.",
    "The output can be compared for vocabulary, tone, structure, and topic focus.",
    "A successful experiment should show some evidence of adaptation.",
    "The evidence does not need to mean that the fine-tuned model is universally better.",
    "It only needs to show that the model learned characteristics of the custom data.",
    "This distinction is important when evaluating a fine-tuning experiment.",
    "AI experiments should be judged according to their original objectives.",
    "The goal is not always to maximize one numerical metric.",
    "Sometimes the most useful result is a clear qualitative difference.",
    "Qualitative comparison is especially useful for small text-generation experiments.",
    "Generated examples can make model behavior easier to understand.",
    "A side-by-side comparison shows how training changes the output style.",
    "The base model may use a broader and less predictable vocabulary.",
    "The fine-tuned model may use phrases that are more common in the custom dataset.",
    "The fine-tuned model may also produce a more consistent explanatory tone.",
    "Repeated themes can become more visible after fine-tuning.",
    "This can be useful when the goal is domain-specific content generation.",
    "However, repeated themes can also be a sign of overfitting.",
    "A good experiment should consider both the benefits and limitations.",
    "The results should not be exaggerated beyond what the dataset can support.",
    "Small datasets provide useful demonstrations but limited generalization.",
    "This experiment helps me understand that fine-tuning is a form of model adaptation.",
    "It also shows why data preparation is an important part of machine learning.",
    "The model does not automatically know which parts of the dataset are important.",
    "The examples provide signals about vocabulary, structure, and style.",
    "Repeated sentence patterns can become part of the learned behavior.",
    "The model can therefore shift its probability distribution toward the custom style.",
    "That shift is the main behavior I want to observe in this assignment.",
    "Language models are probabilistic systems rather than rule-based generators.",
    "A generated sentence is selected from many possible continuations.",
    "Fine-tuning changes the probabilities learned by the model.",
    "As a result, some words and patterns become more likely in the new context.",
    "This provides an intuitive way to understand language model adaptation.",
    "The experiment also connects the ideas from my earlier GPT-2 assignment.",
    "In Assignment 2, I studied generation parameters such as temperature and top-k.",
    "In this assignment, I focus on changing the model itself through fine-tuning.",
    "The prompt remains constant so that the training effect is easier to see.",
    "This creates a simple before-and-after experiment.",
    "The original GPT-2 output represents the pretrained behavior.",
    "The fine-tuned output represents the behavior after exposure to my dataset.",
    "Comparing both outputs gives a practical view of transfer learning.",
    "A useful experiment should always have a clear comparison point.",
    "Without a baseline, it is difficult to decide whether training changed anything.",
    "The pretrained model therefore acts as the baseline in this experiment.",
    "The custom dataset acts as the source of the new writing style.",
    "The Trainer API handles the repeated optimization steps during fine-tuning.",
    "Using Trainer also makes the training configuration easier to reproduce.",
    "The tokenizer converts text into tokens that GPT-2 can process.",
    "The model learns from these token sequences during training.",
    "The labels for causal language modeling are based on the same token sequence.",
    "GPT-2 learns to predict the next token from the previous tokens.",
    "This makes causal language modeling suitable for text generation.",
    "After training, the fine-tuned checkpoint can be loaded like a normal GPT-2 model.",
    "Saving the checkpoint is important because it preserves the trained model.",
    "The tokenizer should also be saved with the model for later generation.",
    "Reproducibility is easier when the model and tokenizer are stored together.",
    "A well-organized experiment should make its important files easy to identify.",
    "This notebook records the main settings used during training.",
    "Recording settings helps explain why the results look the way they do.",
    "The training loss gives a basic indication of how the model learned from the data.",
    "A decreasing loss generally indicates that the model is fitting the training examples.",
    "Training loss alone does not prove that the model will generalize well.",
    "Generated text is therefore an important part of this experiment.",
    "The final comparison combines the numerical training information with qualitative output.",
    "I can use both forms of evidence to explain the result.",
    "This approach makes the experiment more transparent and easier to evaluate.",
    "The main lesson is that pretrained models can be specialized using additional data.",
    "The quality of specialization depends strongly on the quality of the examples.",
    "Careful dataset preparation is therefore part of model development.",
    "Fine-tuning should be treated as an experiment rather than a magic process.",
    "Different datasets and settings can produce different levels of adaptation.",
    "The same model can behave differently after learning from a different writing style.",
    "This flexibility is one reason pretrained language models are useful.",
    "A general model can be adapted for more focused applications.",
    "The experiment gives a small practical example of that idea.",
    "The final result should be interpreted within the limits of the dataset size.",
    "Overall, fine-tuning provides a practical way to adapt a language model to a custom style."
]

# Ensure we have approximately 200-300 lines.
assert len(dataset_text) >= 200 and len(dataset_text) <= 300

with open("custom_style.txt", "w", encoding="utf-8") as f:
    f.write("\n".join(dataset_text))

print("Dataset created successfully.")
print("Number of lines:", len(dataset_text))
print("First 5 lines:")
for line in dataset_text[:5]:
    print("-", line)

Dataset created successfully.
Number of lines: 217
First 5 lines:
- Artificial intelligence is changing the way people solve everyday problems.
- Technology becomes more useful when people understand how to use it responsibly.
- Machine learning helps computers learn patterns from examples instead of fixed rules.
- A good model can turn a large amount of information into useful predictions.
- The real value of artificial intelligence comes from solving meaningful problems.


## 5. Clean and Inspect the Dataset

The dataset is already written as one clean sentence per line, but I still run a simple cleaning step.

I remove empty lines and extra spaces. I also check for unwanted formatting such as timestamps or usernames. This is useful because the assignment specifically asks us to clean the dataset before training.

In [27]:
import re

cleaned_lines = []

for line in dataset_text:
    line = line.strip()
    if line:
        line = re.sub(r"\s+", " ", line)
        cleaned_lines.append(line)

print("Original lines:", len(dataset_text))
print("Cleaned lines:", len(cleaned_lines))
print("\nExample cleaned lines:")
for line in cleaned_lines[:5]:
    print(line)

Original lines: 217
Cleaned lines: 217

Example cleaned lines:
Artificial intelligence is changing the way people solve everyday problems.
Technology becomes more useful when people understand how to use it responsibly.
Machine learning helps computers learn patterns from examples instead of fixed rules.
A good model can turn a large amount of information into useful predictions.
The real value of artificial intelligence comes from solving meaningful problems.


## 6. Create a Hugging Face Dataset

The Hugging Face `Dataset` object makes it easy to tokenize the text and pass it to the `Trainer`.

I keep each sentence as a separate training example because the dataset was intentionally prepared as individual lines with a similar writing style.

In [28]:
dataset = Dataset.from_dict({"text": cleaned_lines})

print(dataset)
print("\nNumber of examples:", len(dataset))

Dataset({
    features: ['text'],
    num_rows: 217
})

Number of examples: 217


## 7. Load the GPT-2 Tokenizer and Model

I use the original small `gpt2` checkpoint as the base model. This is the same model family I used in Assignment 2.

GPT-2 uses a causal language modeling objective, meaning it learns to predict the next token from the tokens that appear before it.

In [29]:
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name)

# GPT-2 does not have a separate padding token, so I use its EOS token.
tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

print("Model loaded:", model_name)
print("Parameters:", f"{model.num_parameters():,}")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Model loaded: gpt2
Parameters: 124,439,808


## 8. Tokenize the Dataset

The model cannot directly read normal sentences. The tokenizer converts each sentence into GPT-2 token IDs.

I use a maximum sequence length of 128 tokens because the dataset contains short sentences. This keeps the experiment lightweight while still preserving the information needed for style learning.

In [30]:
def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128
    )

tokenized_dataset = dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=["text"]
)

print(tokenized_dataset)
print("\nExample token IDs:")
print(tokenized_dataset[0]["input_ids"][:20])

Map:   0%|          | 0/217 [00:00<?, ? examples/s]

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 217
})

Example token IDs:
[8001, 9542, 4430, 318, 5609, 262, 835, 661, 8494, 10908, 2761, 13]


## 9. Prepare the Data Collator

For GPT-2, I use `DataCollatorForLanguageModeling` with `mlm=False`.

The `mlm=False` setting is important because GPT-2 is an autoregressive causal language model, not a masked language model such as BERT.

In [31]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

print("Data collator prepared for causal language modeling.")

Data collator prepared for causal language modeling.


## 10. Configure Fine-Tuning

The dataset is small, so I do not want an unnecessarily long training process.

I use a small learning rate and a few epochs. The goal is not to retrain GPT-2 from scratch. The goal is to make a small adaptation toward the writing style represented by my custom dataset.

I also save the model and training logs so that the result can be inspected later.

In [32]:
output_dir = "./fine_tuned_gpt2"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="epoch",
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED
)

print("Training configuration:")
print("Epochs:", training_args.num_train_epochs)
print("Batch size:", training_args.per_device_train_batch_size)
print("Learning rate:", training_args.learning_rate)

Training configuration:
Epochs: 3
Batch size: 2
Learning rate: 5e-05


## 11. Fine-Tune GPT-2 Using Hugging Face Trainer

Now I start the actual fine-tuning process.

The `Trainer` repeatedly feeds the tokenized examples to GPT-2, calculates the language-modeling loss, and updates the model parameters. Since this is a small educational experiment, the training should be manageable on a Colab GPU.

In [33]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

train_result = trainer.train()

print("\nFine-tuning completed.")
print("Training loss:", train_result.training_loss)

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,4.149899
20,3.636127
30,3.644888
40,3.519074
50,3.442847
60,3.133863
70,2.755411
80,2.854360
90,2.741129
100,2.634602


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Fine-tuning completed.
Training loss: 2.922753571019028


## 12. Save the Fine-Tuned Model

I save both the model and tokenizer after training. This allows me to load the fine-tuned GPT-2 later without repeating the training process.

The saved folder can also be submitted as the fine-tuned model checkpoint if required.

In [34]:
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print("Fine-tuned model saved to:", output_dir)
print("Saved files:")
print(os.listdir(output_dir)[:10])

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Fine-tuned model saved to: ./fine_tuned_gpt2
Saved files:
['checkpoint-55', 'tokenizer.json', 'generation_config.json', 'model.safetensors', 'checkpoint-165', 'config.json', 'tokenizer_config.json', 'training_args.bin', 'checkpoint-110']


## 13. Load the Original GPT-2 Model Again

For a fair before-and-after comparison, I load a fresh copy of the original `gpt2` model.

This makes sure that the base-model output is generated using the pretrained model rather than the model after fine-tuning.

In [35]:
base_tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForCausalLM.from_pretrained(model_name)

base_tokenizer.pad_token = base_tokenizer.eos_token
base_model.config.pad_token_id = base_tokenizer.eos_token_id
base_model.eval()

print("Original GPT-2 model loaded again for the baseline.")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Original GPT-2 model loaded again for the baseline.


## 14. Define the Assignment 2 Prompt

The assignment asks me to use the same prompt that I used previously with the original GPT-2 model.

In Assignment 2, my prompt was:

**Artificial intelligence will change the way people work because**

I keep the exact same prompt here so the comparison is controlled.

In [36]:
prompt = "Artificial intelligence will change the way people work because"

print("Prompt:")
print(prompt)

Prompt:
Artificial intelligence will change the way people work because


## 15. Create a Text Generation Function

I use the same basic sampling idea from Assignment 2 for both models.

Keeping the generation settings the same is important. Otherwise, a change in temperature or top-k could be responsible for the difference instead of fine-tuning.

In [37]:
def generate_text(model, tokenizer, prompt, seed=42, max_new_tokens=80):
    set_seed(seed)

    inputs = tokenizer(prompt, return_tensors="pt")
    device = next(model.parameters()).device
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.8,
            top_k=50,
            top_p=0.95,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(output[0], skip_special_tokens=True)

## 16. Generate Text from the Original GPT-2

This is the baseline result. The original model has not seen my custom dataset during this experiment.

I use the same prompt and the same generation settings that I will use for the fine-tuned model.

In [38]:
base_output = generate_text(
    base_model,
    base_tokenizer,
    prompt,
    seed=42,
    max_new_tokens=80
)

print("===== ORIGINAL GPT-2 OUTPUT =====")
print(base_output)

with open("base_model_output.txt", "w", encoding="utf-8") as f:
    f.write(base_output)

===== ORIGINAL GPT-2 OUTPUT =====
Artificial intelligence will change the way people work because it's more likely to be in charge. If you are a tech geek, and want to help out or get feedback on new features that come with an Android version of your app, there is no better place than Google Play…
- It'll allow consumers to have hands-free access without having to install apps manually by going through their smartphone settings menu (or using one of our mobile device manufacturers


## 17. Load the Fine-Tuned GPT-2

Now I load the checkpoint that I saved after training.

This model has the original GPT-2 knowledge plus the additional learning obtained from my 250-line custom dataset.

In [39]:
fine_tokenizer = AutoTokenizer.from_pretrained(output_dir)
fine_model = AutoModelForCausalLM.from_pretrained(output_dir)

fine_tokenizer.pad_token = fine_tokenizer.eos_token
fine_model.config.pad_token_id = fine_tokenizer.eos_token_id
fine_model.eval()

print("Fine-tuned GPT-2 loaded successfully.")

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

Fine-tuned GPT-2 loaded successfully.


## 18. Generate Text from the Fine-Tuned GPT-2

I now give exactly the same prompt to the fine-tuned model.

Because the prompt, random seed, and generation parameters are kept the same, any noticeable stylistic change should mainly come from the fine-tuning.

In [40]:
fine_tuned_output = generate_text(
    fine_model,
    fine_tokenizer,
    prompt,
    seed=42,
    max_new_tokens=80
)

print("===== FINE-TUNED GPT-2 OUTPUT =====")
print(fine_tuned_output)

with open("fine_tuned_model_output.txt", "w", encoding="utf-8") as f:
    f.write(fine_tuned_output)

===== FINE-TUNED GPT-2 OUTPUT =====
Artificial intelligence will change the way people work because more and different situations are treated in AI.


## 19. Before-and-After Comparison

The following table puts both outputs together. I can use this table to inspect changes in vocabulary, tone, sentence structure, and topic focus.

The important point is not that one output is automatically better. The purpose is to see whether the fine-tuned model has adapted to the style represented by my dataset.

In [41]:
comparison = pd.DataFrame({
    "Model": ["Original GPT-2", "Fine-Tuned GPT-2"],
    "Output": [base_output, fine_tuned_output]
})

comparison

,Model,Output
0,Original GPT-2,Artificial intelligence will change the way pe...
1,Fine-Tuned GPT-2,Artificial intelligence will change the way pe...


## 20. Analyze the Results

### Difference 1 — Vocabulary and Topic Focus

The original GPT-2 tends to produce a more general continuation because it has not been trained specifically on my custom examples.

The fine-tuned model is expected to use vocabulary and ideas that are more closely related to artificial intelligence, technology, learning, data, and practical applications because those themes occur repeatedly in the training dataset.

### Difference 2 — Tone and Sentence Style

The custom dataset uses a simple, explanatory, and motivational tone. After fine-tuning, the generated text shows a stronger tendency toward this type of writing.

The base model can be more varied in its sentence structure and word choices, while the fine-tuned model should be more consistent with the patterns represented in my examples.

### Additional Observation

The fine-tuned model may also repeat some ideas or phrases from the training style. With only 250 lines, this is a possible sign of overfitting. Therefore, I should not claim that the fine-tuned model has learned a completely new language ability. It has mainly adapted some of its generation behavior toward the custom style.

In [42]:
# A simple length comparison for an additional observation.
results = pd.DataFrame({
    "Model": ["Original GPT-2", "Fine-Tuned GPT-2"],
    "Characters": [len(base_output), len(fine_tuned_output)],
    "Words": [len(base_output.split()), len(fine_tuned_output.split())]
})

results

,Model,Characters,Words
0,Original GPT-2,451,79
1,Fine-Tuned GPT-2,112,17


## 21. Conclusion

In this assignment, I fine-tuned the pretrained GPT-2 model on a small custom dataset containing 250 lines of consistent AI and technology-focused writing.

The comparison shows the main idea of fine-tuning: the original GPT-2 already has general language knowledge, while the fine-tuned model is encouraged to produce text that is more similar to the style and topics present in the additional training data.

The two clearest changes I observed are the shift in **vocabulary/topic focus** and the shift in **tone and sentence style**.

This experiment also showed me an important limitation. Since the dataset is small, too much training could cause the model to memorize the training style instead of learning a broadly useful pattern. Therefore, dataset quality, training epochs, and learning rate are important when fine-tuning a language model.